# Exp 05: Control Flow

目标：
1. 区分 data-independent 和 data-dependent 控制流。
2. 理解 `.item()` / `bool(tensor)` 为什么容易触发 graph break。
3. 掌握 `torch.cond`、`torch.compiler.is_compiling()`、隔离策略三类处理方式。
4. 用 `torch._dynamo.explain` 快速查看 graph break 数量和原因。

推荐脚本版配合日志运行：

```bash
TORCH_LOGS=graph_breaks python experiments_py/exp05_control_flow.py
TORCH_LOGS=graph_code python experiments_py/exp05_control_flow.py
```

## 核心判断

- data-independent 分支：依赖 shape、dtype、module config、`self.training` 等编译时可知信息。Dynamo 会生成 guard 并 specialize，通常不 graph break。
- data-dependent 分支：依赖 tensor 运行时数值。Python 解释器必须知道真假，因此 `.item()` 或 `if tensor:` 会把值同步回 CPU，导致 graph break。

如果 data-dependent 分支在核心热路径里，应该考虑 `torch.cond` 或把外层控制逻辑隔离在 eager 中。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DTYPE = torch.bfloat16
BATCH = 32
DIM = 512

torch.manual_seed(42)

In [ ]:
def explain_graph_breaks(fn, *args) -> dict[str, object]:
    explanation = torch._dynamo.explain(fn)(*args)
    return {
        "graph_count": explanation.graph_count,
        "graph_break_count": explanation.graph_break_count,
        "op_count": explanation.op_count,
        "break_reasons": [str(reason) for reason in explanation.break_reasons],
    }


def benchmark_forward(fn, x: torch.Tensor, repeats: int = 30) -> float:
    for _ in range(5):
        fn(x)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn(x)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


x = torch.randn(BATCH, DIM, device=DEVICE, dtype=DTYPE)

## Case A：data-independent 分支

下面的 `if x.shape[-1] == DIM` 和 `if x.dtype == ...` 都能在编译期确定。Dynamo 会为这些条件生成 guard，如果后续输入不满足 guard，才会重新编译。

In [ ]:
class DataIndependentBranch(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(DIM, DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] == DIM:
            x = F.gelu(self.fc(x))
        else:
            x = x * 2.0

        if x.dtype == torch.bfloat16:
            x = x.float()
        return x.mean(dim=-1)


model_a = DataIndependentBranch().to(DEVICE, dtype=DTYPE)
info_a = explain_graph_breaks(model_a, x)
compiled_a = torch.compile(model_a, fullgraph=False)
time_a = benchmark_forward(compiled_a, x)

print(info_a)
print(f"compiled forward: {time_a:.3f} ms")

## Case B：data-dependent 分支

`.item()` 把 GPU tensor 的值取回 Python。Dynamo 无法在编译时知道这个值，因此会在这里切断 graph。

graph break 的代价不是只有“少编译一段”：中间 tensor 会被实体化，跨 break 的 fusion、CUDA Graph 捕获、backward 合并机会都可能丢失。

In [ ]:
class DataDependentBranch(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(DIM, DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc(x)
        if h.sum().item() > 0:
            h = F.relu(h)
        else:
            h = F.gelu(h)
        return h.mean(dim=-1)


model_b = DataDependentBranch().to(DEVICE, dtype=DTYPE)
info_b = explain_graph_breaks(model_b, x)
compiled_b = torch.compile(model_b, fullgraph=False)
time_b = benchmark_forward(compiled_b, x)

print({k: v for k, v in info_b.items() if k != "break_reasons"})
print("first break reason:", info_b["break_reasons"][:1])
print(f"compiled forward: {time_b:.3f} ms")

## Case C：`torch.cond`

`torch.cond` 把 data-dependent 分支表示成高阶算子，两条分支会作为子图被捕获，因此可以保持单图。

限制也更严格：true/false 分支的输入输出结构、shape、dtype 必须一致；分支里尽量避免 mutation 和 Python 副作用。训练场景还需要针对当前 PyTorch 版本做回归测试。

In [ ]:
class CondBranch(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(DIM, DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc(x)
        pred = h.sum() > 0
        h = torch.cond(pred, lambda t: F.relu(t), lambda t: F.gelu(t), (h,))
        return h.mean(dim=-1)


model_c = CondBranch().to(DEVICE, dtype=DTYPE)
info_c = explain_graph_breaks(model_c, x)
compiled_c = torch.compile(model_c, fullgraph=True)
time_c = benchmark_forward(compiled_c, x)

print(info_c)
print(f"compiled forward: {time_c:.3f} ms")

## Case D/E：调试副作用与隔离策略

`torch.compiler.is_compiling()` 在编译时返回 `True`，适合保护 `assert`、`print`、NaN 检查等调试副作用，让它们只在 eager 路径运行。

如果外层控制流确实必须由 Python 决策，可以把外层留在 eager，只把内部纯 tensor 计算编译。这个隔离策略保守但稳定。

In [ ]:
class DebugFriendlyModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(DIM, DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.compiler.is_compiling():
            assert x.dim() == 2, f"expected 2D input, got {tuple(x.shape)}"
            if torch.isnan(x).any():
                print("NaN detected in input")

        h = F.gelu(self.fc(x))
        return h.mean(dim=-1)


@torch.compile(fullgraph=True)
def compiled_inner(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return F.linear(x, weight, bias).relu()


def eager_router_with_compiled_inner(x: torch.Tensor, fc: nn.Linear) -> torch.Tensor:
    if x.sum().item() > 0:
        out = compiled_inner(x, fc.weight, fc.bias)
    else:
        out = compiled_inner(-x, fc.weight, fc.bias)
    return out.mean(dim=-1)


model_d = DebugFriendlyModel().to(DEVICE, dtype=DTYPE)
info_d = explain_graph_breaks(model_d, x)
time_d = benchmark_forward(torch.compile(model_d, fullgraph=True), x)

fc = nn.Linear(DIM, DIM).to(DEVICE, dtype=DTYPE)
time_e = benchmark_forward(lambda inp: eager_router_with_compiled_inner(inp, fc), x)

print("debug-friendly:", info_d, f"time={time_d:.3f} ms")
print(f"isolated router time={time_e:.3f} ms")

In [ ]:
rows = [
    ("data-independent", info_a["graph_break_count"], time_a),
    ("data-dependent", info_b["graph_break_count"], time_b),
    ("torch.cond", info_c["graph_break_count"], time_c),
    ("is_compiling", info_d["graph_break_count"], time_d),
    ("eager router + compiled inner", 0, time_e),
]

print(f"{'case':32s} {'breaks':>8s} {'fwd ms':>10s}")
print("-" * 54)
for name, breaks, elapsed_ms in rows:
    print(f"{name:32s} {breaks:8d} {elapsed_ms:10.3f}")

## 小结

控制流处理建议：

1. 分支依赖 shape、dtype、config、training mode：直接写，Dynamo 会 guard + specialize。
2. 分支只是调试代码：用 `torch.compiler.is_compiling()` 保护。
3. 少量非核心 data-dependent 分支：可以先容忍 graph break，并用 `TORCH_LOGS=graph_breaks` 观察影响。
4. 核心热路径中的 data-dependent 分支：优先尝试 `torch.cond`，但要确认分支输出一致并做训练回归测试。
5. 外层调度复杂、内层计算稳定：外层 eager，内层 `torch.compile(fullgraph=True)`。